In [17]:
# Setup: imports and environment checks
import os
import sys
from pathlib import Path

# Ensure project paths
ROOT = Path(r"c:\Users\junhongs\Desktop\capstone\evaluation")
PDF_PATH = ROOT / "material" / "UAS-OATH-Token-Implementation-Guide-6.0.1-GA (AI).pdf"
OUTPUT_DIR = ROOT / "dataset" /"uas_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDF exists: {PDF_PATH.exists()} -> {PDF_PATH}")

# Soft dependency checks
missing = []
for pkg in ["ragas", "langchain_community", "langchain_openai", "openai", "pypdf", "pandas"]:
    try:
        __import__(pkg)
    except Exception:
        missing.append(pkg)

if missing:
    print("Missing packages detected:\n - " + "\n - ".join(missing))
    print("Install them in this kernel, for example:")
    print("%pip install ragas langchain-community langchain-openai openai pypdf pandas tqdm")
else:
    print("All required packages found.")

PDF exists: True -> c:\Users\junhongs\Desktop\capstone\evaluation\material\UAS-OATH-Token-Implementation-Guide-6.0.1-GA (AI).pdf
All required packages found.


In [ ]:
# Load and chunk the PDF into LangChain Documents
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

assert PDF_PATH.exists(), f"PDF not found at {PDF_PATH}"

loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()
print(f"Loaded {len(docs)} page-level documents")

# Chunk to ~800 tokens equivalent (~3000-4000 chars) for better synthesis
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", ".", " "]
)
chunked_docs = text_splitter.split_documents(docs)
print(f"Chunked into {len(chunked_docs)} documents")

# Show a preview
for i, d in enumerate(chunked_docs[:2]):
    print(f"Chunk {i} | {len(d.page_content)} chars | meta: {d.metadata}")

Loaded 85 page-level documents
Chunked into 93 documents
Chunk 0 | 49 chars | meta: {'producer': 'Developer Express Inc. DXperience (tm) v24.1.7', 'creator': 'Microsoft Office Word', 'creationdate': '2025-09-11T04:33:00+00:00', 'title': 'UAS-OATH-Token-Implementation-Guide-6.0.1-GA (AI)', 'author': 'ClickHelp.com', 'moddate': '2025-09-11T04:33:00+00:00', 'source': 'c:\\Users\\junhongs\\Desktop\\capstone\\evaluation\\material\\UAS-OATH-Token-Implementation-Guide-6.0.1-GA (AI).pdf', 'total_pages': 85, 'page': 0, 'page_label': '1'}
Chunk 1 | 1733 chars | meta: {'producer': 'Developer Express Inc. DXperience (tm) v24.1.7', 'creator': 'Microsoft Office Word', 'creationdate': '2025-09-11T04:33:00+00:00', 'title': 'UAS-OATH-Token-Implementation-Guide-6.0.1-GA (AI)', 'author': 'ClickHelp.com', 'moddate': '2025-09-11T04:33:00+00:00', 'source': 'c:\\Users\\junhongs\\Desktop\\capstone\\evaluation\\material\\UAS-OATH-Token-Implementation-Guide-6.0.1-GA (AI).pdf', 'total_pages': 85, 'page': 1, 'pag

In [19]:
# Verify API key is visible to this kernel and optionally load from .env
import os

# Optional: auto-load from a .env file if present
try:
    from dotenv import load_dotenv  # type: ignore
    loaded = load_dotenv()
    if loaded:
        print("Loaded environment from .env")
except Exception:
    pass  # python-dotenv not installed; that's okay

api_key = os.environ.get("OPENAI_API_KEY", "")
masked = (api_key[:4] + "***" + api_key[-4:]) if api_key else None
print("OPENAI_API_KEY set:", bool(api_key), f"({masked})" if masked else "(None)")

# Tip for VS Code/Jupyter:
# If you set $env:OPENAI_API_KEY in a separate PowerShell window, you may need to
# restart the Jupyter kernel so this process sees the new environment.

Loaded environment from .env
OPENAI_API_KEY set: True (sk-s***mWMA)


In [ ]:
domain_prompt = """
### 1. ROLE AND GOAL
You are an Software Engineer and Test Set Generator at 'i-sprint innovations.' Your goal is to create a "golden" evaluation dataset for a new RAG system. This dataset will test the RAG's ability to provide accurate, grounded technical support to developers and deployment engineers working with the i-sprint product suite (like UAS).

### 2. TASK
You will be provided with a technical document chunk. Your task is to generate 3-5 high-quality, complex question-answer pairs based *exclusively* on this text. You must output a valid JSON list.

### 3. GENERATION RULES
- **100% GROUNDED:** The `question` must be answerable *only* with the provided text. The `ground_truth_answer` must be a concise, factual summary of the answer. The `source_context` must be the *exact quote(s)* from the text that support the answer.
- **PERSONA-DRIVEN:** Questions must sound like they are from a technical engineer. Use professional, specific, and concise phrasing.
- **TOPIC FOCUS:** Questions must target key identity and security concepts: authentication flows, identity federation (SAML, OIDC), authorization policies, OATH token implementation, security configurations, deployment steps, or troubleshooting error codes.
- **DIVERSE SCENARIOS:** You must generate questions from at least two of the following `question_type` categories:
    1.  **Deployment/Configuration:** Questions about setup, high-availability, or setting parameters.
    2.  **Troubleshooting/Error Handling:** Questions about resolving errors, log analysis, or audit procedures.
    3.  **Integration/Development:** Questions about API endpoints, coding standards, or integrating with protocols like OAuth2.

Example could be : 
Question : How maximum number of users is checked in AccessMatrix 5.x?
Answer: It is done by counting number of users in internal user store (default user store and external user stores). So the checking is based on an estimation of number of unique user login events for last 3 months by ust_last_login_date in am_userauthstate table. This is done by a scheduled housekeeping task (defaulted to run on everyday).Hence, the checking does not run this query every time it is run. Instead, it will read this cached information from the database.If licence is violated, Admin Console only allows update of server licence key. In this case, only Admin Console is affected while other non-admin users will not be affected where they can still perform normal login and ESSO functions. Deleting users will not help because the counting is based on last 3 months login events, and not on number of users.
ground truth: "The Max Users value refers to allowable user counts in a system. This user count refers to the login users of the system, not the total users created or the total users existing in the various user stores. Max Users checking is based on a figure derived from the estimation of distinct active login users during the last 3 months.",  "The database table am_userauthstate stores changes on user authentication states. Within the am_userauthstate table schema, the column ust_last_login_date records the Last login date.",  "The Housekeeping module is used to housekeep data or records in the database, and housekeeping tasks can be configured to run based on a user-defined schedule. The housekeeping task execution interval is defaulted to 24 hours", "If the derived figure exceeds Max Users specified in the license, the License violation prompt is triggered where the AccessMatrix Admin Console will only present a restricted menu until a new valid license is applied",   "If the derived figure exceeds Max Users specified in the license, the License violation prompt is triggered where the AccessMatrix Admin Console will only present a restricted menu until a new valid license is applied", "If a license violation occurs, Normal user functions like user login and all API functions will continue to work properly as usual.",  "The counting refers to the login users of the system, not the total users created or the total users existing in the various user stores."

### 4. OUTPUT FORMAT
Respond ONLY with a valid JSON list. Do not include any text before or after the JSON.

[
  {
    "question_type": "Deployment/Configuration | Troubleshooting/Error Handling | Integration/Development",
    "question": "A specific, professional question based *only* on the document.",
    "ground_truth_answer": "A concise, factual answer derived *only* from the document.",
    "source_context": "The exact quote or passage from the document that contains the answer."
  },
  {
    "question_type": "...",
    "question": "...",
    "ground_truth_answer": "...",
    "source_context": "..."
  }
]

"""


In [ ]:
class Persona:
    def __init__(self, name: str, role_description: str):
        self.name = name
        self.role_description = role_description
personas = [
    Persona(
        name="Technical Analyst",
        role_description="Focuses on detailed system specifications and API documentation"
    ),
    Persona(
        name="Novice User",
        role_description="Asks simple questions using layman terms and basic functionality"
    )
]

In [ ]:
# Require OpenAI credentials via environment
if not os.environ.get("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY not set."
    )



In [ ]:
# # Configure LLMs and Embeddings for Ragas test generation
# import os
# import openai
# from ragas.llms import LangchainLLMWrapper
# from ragas.embeddings import OpenAIEmbeddings
# from langchain_openai import ChatOpenAI
# from ragas.testset import TestsetGenerator
# from ragas.llms.base import llm_factory;

# # Require OpenAI credentials via environment
# if not os.environ.get("OPENAI_API_KEY"):
#     raise EnvironmentError(
#         "OPENAI_API_KEY not set. In PowerShell: $env:OPENAI_API_KEY = 'sk-...'"
#     )




# default_llm = llm_factory("gpt-4o-mini")
# openai_client = openai.OpenAI()  # uses env creds
# emb_model = OpenAIEmbeddings(client=openai_client)

# # Initialize generator
# generator = TestsetGenerator(llm=default_llm, embedding_model=emb_model)

# # Generate a small test set first to sanity check
# TESTSET_SIZE = 1
# print(f"Generating {TESTSET_SIZE} synthetic queries...")
# dataset = generator.generate_with_langchain_docs(
#     documents=chunked_docs,
#     testset_size=TESTSET_SIZE,
#     prompt=domain_prompt
# )
# print("Done.")

# df = dataset.to_pandas()
# print(df.head())

In [ ]:

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

# Define the model explicitly (e.g., gpt-4o for high quality generation)
openai_model = "gpt-4o" 

generator_llm = LangchainLLMWrapper(ChatOpenAI(model=openai_model, temperature=0.1))  
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    persona_list=personas
)

# Define specific distribution to ensure test hardness
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.2)
]

dataset = generator.generate_with_langchain_docs(
    chunked_docs, # Use chunked_docs, not docs (which are full pages)
    testset_size=1# filepath: c:\Users\junhongs\Desktop\capstone\evaluation\dataLoader_ragas.ipynb
# ...existing code...
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

# Define the model explicitly (e.g., gpt-4o for high quality generation)
openai_model = "gpt-4o" 

generator_llm = LangchainLLMWrapper(ChatOpenAI(model=openai_model, temperature=0.1))  
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    persona_list=personas
)

# Define specific distribution to ensure test hardness
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.2)
]

dataset = generator.generate_with_langchain_docs(
    chunked_docs, # Use chunked_docs, not docs (which are full pages)
    testset_size=10, # Start small to test
    query_distribution=query_distribution
)


In [ ]:
# Add this as a new code cell after defining domain_prompt
from datasets import Dataset

answer_prompt = """
Answer the question using only the provided document content. Avoid hallucination.
Provide a concise, accurate, and technically detailed response for a software engineer.
"""

# Convert LangChain Documents -> HF Dataset with a 'text' field
hf_docs = Dataset.from_list([{"text": d.page_content, "metadata": d.metadata} for d in chunked_docs])

# Use the customizable API (available in newer ragas versions)
try:
    dataset = generator.generate(
        documents=hf_docs,
        num_questions=1,
        question_prompt_template=domain_prompt,
        answer_prompt_template=answer_prompt,
    )
except TypeError:
    # Fallback for older ragas: no custom templates available in this path
    print("generator.generate(...) with custom templates not supported in this ragas version.")
    print("Falling back to generate_with_langchain_docs without custom prompt.")
    dataset = generator.generate_with_langchain_docs(
        documents=chunked_docs,
        testset_size=10,
        prompt=domain_prompt,
    )

df = dataset.to_pandas()
print(df.head())

generator.generate(...) with custom templates not supported in this ragas version.
Falling back to generate_with_langchain_docs without custom prompt.


Applying CustomNodeFilter:   0%|          | 0/93 [00:00<?, ?it/s]Node d614fedf-93ee-41b3-9223-432a57c7873d does not have a summary. Skipping filtering.
Node fecf477b-2f7b-4adf-85c6-d18e1642e54d does not have a summary. Skipping filtering.
Node 75f4e3ab-3da0-402f-8f5a-4f6e3fdde021 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:   4%|▍         | 4/93 [00:02<00:54,  1.64it/s]Node dc4db8a3-c8b4-4bac-adfa-bd9def60bd21 does not have a summary. Skipping filtering.
Node 36e110ec-a9a6-41cf-a5ec-fc5ba2d50245 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:  45%|████▌     | 42/93 [01:08<01:10,  1.38s/it]Node 2502e646-b971-4306-8da7-0cb43beec778 does not have a summary. Skipping filtering.
Node a967ea2b-568b-4f78-bdc3-2ae825afde12 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:  67%|██████▋   | 62/93 [01:40<00:38,  1.25s/it]Node 2a0b2a2c-44af-4beb-920a-e405dfca0b7a does not have a summary. Skipping filtering.
Node 7ea90a69-a306

                                          user_input  \
0  How does the UAS OATHTokenImplementation Guide...   
1  What is HSM doing in UAS OATHFM and how it hel...   
2                          What AccessMatrix UAS do?   
3  What are the benefits of RFC6287 in OATH-based...   
4  How can the token app be utilized for token ma...   

                                  reference_contexts  \
0  [UAS OATHTokenImplementation Guide - 6.0.1-GA ...   
1  [Tableof Contents\n1. Preface - UAS OATHTokenI...   
2  [1. Preface - UAS OATHTokenImplementation Guid...   
3  [2. Introduction to UAS OATHTokenImplementatio...   
4  [<1-hop>\n\ntoken app (e.g. via QR or retrieve...   

                                           reference  \
0  The UAS OATHTokenImplementation Guide - 6.0.1-...   
1  HSM is mentioned in the context of UAS OATHFM,...   
2  AccessMatrix UAS provides detailed information...   
3  RFC6287 is part of the standardized specificat...   
4  The token app can be utilized for token man

In [24]:
df.head(2)

,user_input,reference_contexts,reference,synthesizer_name
0,How does the UAS OATHTokenImplementation Guide...,[UAS OATHTokenImplementation Guide - 6.0.1-GA ...,The UAS OATHTokenImplementation Guide - 6.0.1-...,single_hop_specific_query_synthesizer
1,What is HSM doing in UAS OATHFM and how it hel...,[Tableof Contents\n1. Preface - UAS OATHTokenI...,"HSM is mentioned in the context of UAS OATHFM,...",single_hop_specific_query_synthesizer


In [25]:
# Save dataset in JSONL and CSV for cross-framework evaluations
import json
import pandas as pd
import numpy as np

# Inspect available columns
print("Columns:", list(df.columns))

def _is_nonempty_value(x):
    if x is None:
        return False
    if isinstance(x, str):
        return x.strip() != ""
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) > 0
    if isinstance(x, np.ndarray):
        return x.size > 0
    try:
        import math
        if isinstance(x, float) and math.isnan(x):
            return False
    except Exception:
        pass
    return True
    
# Try to harmonize to a simple schema: query, ground_truth, contexts(list[str])
def infer_columns(row):
    # query
    query = None
    for k in ["user_input", "question", "query", "prompt"]:
        if k in row and pd.notna(row[k]):
            query = row[k]
            break
    # ground truth answer
    gt = None
    for k in ["reference", "ground_truth", "expected_output", "answer"]:
        if k in row:
            val = row[k]
            if _is_nonempty_value(val):
                gt = val
                break
    # contexts
    contexts = None
    for k in ["contexts", "reference_contexts", "contexts_text", "documents"]:
        if k in row:
            val = row[k]
            if not _is_nonempty_value(val):
                continue
            # Ensure list[str]
            if isinstance(val, str):
                contexts = [val]
            elif isinstance(val, (list, tuple)):
                # Some entries might be dicts with 'page_content'
                first = val[0] if len(val) else None
                if isinstance(first, dict) and "page_content" in first:
                    contexts = [d.get("page_content", "") for d in val]
                else:
                    contexts = [str(v) for v in val]
            elif isinstance(val, np.ndarray):
                if val.size: 
                    first = val.flat[0]
                    if isinstance(first, dict) and "page_content" in first:
                        contexts = [d.get("page_content", "") for d in val.tolist()]
                    else:
                        contexts = [str(v) for v in val.tolist()]
            elif isinstance(val, dict) and "page_content" in val:
                contexts = [val.get("page_content", "")]            
            else:
                contexts = [str(val)]
            break
    return query, gt, contexts or []

records = []
for _, row in df.iterrows():
    q, gt, ctx = infer_columns(row)
    records.append({
        "query": q,
        "ground_truth": gt,
        "contexts": ctx,
    })

# File paths
jsonl_path = OUTPUT_DIR / "synthetic_usoadmin_ragas.jsonl"
csv_path = OUTPUT_DIR / "synthetic_usoadmin_ragas.csv"

# Write JSONL
with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Write CSV (flatten contexts)
pd.DataFrame({
    "query": [r["query"] for r in records],
    "ground_truth": [r["ground_truth"] for r in records],
    "contexts_joined": ["\n\n".join(r["contexts"]) for r in records]
}).to_csv(csv_path, index=False, encoding="utf-8")

print("Saved:")
print(" -", jsonl_path)
print(" -", csv_path)

Columns: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']
Saved:
 - c:\Users\junhongs\Desktop\capstone\evaluation\dataset\ragas_generated_testset\uas_dataset\synthetic_usoadmin_ragas.jsonl
 - c:\Users\junhongs\Desktop\capstone\evaluation\dataset\ragas_generated_testset\uas_dataset\synthetic_usoadmin_ragas.csv


In [27]:
# Convert dataset/UAS-OATH.jsonl (JSON array or JSON Lines) to CSV
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

# Expect ROOT defined earlier in the notebook
json_path = ROOT / "dataset" / "UAS-OATH.json"
csv_path = ROOT / "dataset" / "UAS-OATH.csv"

assert json_path.exists(), f"JSON/JSONL file not found: {json_path}"

text = json_path.read_text(encoding="utf-8").strip()

# Try parsing as JSON array; fallback to JSONL
try:
    if text.startswith("["):
        data = json.loads(text)
    else:
        raise ValueError("Not a JSON array; try JSONL")
except Exception:
    # JSON Lines (one JSON object per line); ignore empty lines and '//' comments
    data = [json.loads(line) for line in text.splitlines() if line.strip() and not line.strip().startswith("//")]

# Ensure list of dicts
if isinstance(data, dict):
    data = [data]
assert isinstance(data, list), "Parsed data must be a list of objects"

# DataFrame
df_json = pd.DataFrame(data)

# Order columns if present
preferred_cols = ["question_type", "question", "ground_truth_answer", "source_context"]
cols = [c for c in preferred_cols if c in df_json.columns] + [c for c in df_json.columns if c not in preferred_cols]
if cols:
    df_json = df_json[cols]

# Write CSV
df_json.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Wrote {len(df_json)} rows to: {csv_path}")

# Preview
display(df_json.head(3))

Wrote 10 rows to: c:\Users\junhongs\Desktop\capstone\evaluation\dataset\UAS-OATH.csv


,question_type,question,ground_truth_answer,source_context
0,Deployment/Configuration,What configuration setting in the Global Setti...,The 'Randomize Reservation Pool' should be set...,Randomize Reservation Pool This is to randomiz...
1,Integration/Development,When utilizing the `/token/OATH/create-assign-...,The token seed is first encrypted using RSA wi...,The application needs to generate a key pair a...
2,Troubleshooting/Error Handling,What is the physical cause mentioned in the so...,The internal clock may drift due to environmen...,OTP token internal clock may drift (for enviro...


In [31]:
# Convert dataset/UAS-OATH.jsonl (JSON array or JSON Lines) to CSV
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

# Expect ROOT defined earlier in the notebook
json_path = ROOT / "dataset" / "AM-HSM.json"
csv_path = ROOT / "dataset" / "AM-HSM.csv"

assert json_path.exists(), f"JSON/JSONL file not found: {json_path}"

text = json_path.read_text(encoding="utf-8").strip()

# Try parsing as JSON array; fallback to JSONL
try:
    if text.startswith("["):
        data = json.loads(text)
    else:
        raise ValueError("Not a JSON array; try JSONL")
except Exception:
    # JSON Lines (one JSON object per line); ignore empty lines and '//' comments
    data = [json.loads(line) for line in text.splitlines() if line.strip() and not line.strip().startswith("//")]

# Ensure list of dicts
if isinstance(data, dict):
    data = [data]
assert isinstance(data, list), "Parsed data must be a list of objects"

# DataFrame
df_json = pd.DataFrame(data)

# Order columns if present
preferred_cols = ["question_type", "question", "ground_truth_answer", "source_context"]
cols = [c for c in preferred_cols if c in df_json.columns] + [c for c in df_json.columns if c not in preferred_cols]
if cols:
    df_json = df_json[cols]

# Write CSV
df_json.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Wrote {len(df_json)} rows to: {csv_path}")

# Preview
display(df_json.head(3))

Wrote 10 rows to: c:\Users\junhongs\Desktop\capstone\evaluation\dataset\AM-HSM.csv


,question_type,question,ground_truth_answer,source_context
0,Deployment/Configuration,Edge Case — What is the unsupported deployment...,On-premise deployment of AccessMatrix that int...,AccessMatrix must be deployed in Linux VM on G...
1,Integration/Development,What authentication strategy does AccessMatrix...,AccessMatrix uses the Application Default Cred...,AccessMatrix uses Application Default Credenti...
2,Deployment/Configuration,What three specific GCP configuration paramete...,The administrator must specify the Key Ring Na...,"5. Specify the Key Ring Name (e.g. ""ISPRINT_KR..."
